# MATH 5010 Computer Lab — Section 7  
## Inequalities and Identities  
### Full Solutions Included

This lab accompanies **Section 7: Inequalities and Identities**.

We will use Python to explore:

1. Boole's inequality and inclusion-exclusion  
2. Bonferroni bounds  
3. Markov's inequality  
4. Chebyshev's inequality  
5. Chernoff bounds  
6. Hoeffding's inequality  
7. Cauchy--Schwarz inequality and correlation  
8. Hölder's inequality  
9. Minkowski's inequality  
10. Jensen's inequality  
11. Practice problems with complete solutions

The main idea is that probability inequalities provide useful bounds even when exact probabilities are hard to compute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from math import comb, exp, log, sqrt

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 5)
print("Packages loaded.")

## 1. Boole's Inequality

Boole's inequality, also called the union bound, says

\[
P\left(\bigcup_{i=1}^n E_i\right)
\le
\sum_{i=1}^n P(E_i).
\]

It is useful because the probability of a union may be hard to compute, but the individual probabilities may be easy.

In [ ]:
# Simulate three events on a probability space.
N = 300_000
U = rng.uniform(0, 1, size=N)
V = rng.uniform(0, 1, size=N)

E1 = U < 0.30
E2 = V < 0.40
E3 = U + V < 0.50

p_union = np.mean(E1 | E2 | E3)
sum_probs = np.mean(E1) + np.mean(E2) + np.mean(E3)

print("P(E1 union E2 union E3):", p_union)
print("P(E1)+P(E2)+P(E3):", sum_probs)
print("Boole inequality holds:", p_union <= sum_probs)

### Solution

The simulation verifies

\[
P(E_1\cup E_2\cup E_3)\le P(E_1)+P(E_2)+P(E_3).
\]

The inequality may be loose because the events can overlap.

## 2. Inclusion-Exclusion Formula

For three events,

\[
P(E_1\cup E_2\cup E_3)
=
S_1-S_2+S_3,
\]

where

\[
S_1=\sum_i P(E_i),
\qquad
S_2=\sum_{i<j}P(E_i\cap E_j),
\qquad
S_3=P(E_1\cap E_2\cap E_3).
\]

In [ ]:
p1, p2, p3 = np.mean(E1), np.mean(E2), np.mean(E3)

p12 = np.mean(E1 & E2)
p13 = np.mean(E1 & E3)
p23 = np.mean(E2 & E3)
p123 = np.mean(E1 & E2 & E3)

S1 = p1 + p2 + p3
S2 = p12 + p13 + p23
S3 = p123

inclusion_exclusion = S1 - S2 + S3

table = pd.DataFrame({
    "quantity": ["P(union)", "S1", "S2", "S3", "S1 - S2 + S3"],
    "value": [p_union, S1, S2, S3, inclusion_exclusion]
})
display(table)

### Solution

The exact formula subtracts pairwise overlaps and then adds back the triple overlap:

\[
P(E_1\cup E_2\cup E_3)
=
P(E_1)+P(E_2)+P(E_3)
-P(E_1E_2)-P(E_1E_3)-P(E_2E_3)
+P(E_1E_2E_3).
\]

The simulated value of $S_1-S_2+S_3$ should match the direct union probability.

## 3. Bonferroni Bounds

The Bonferroni inequalities give partial inclusion-exclusion bounds.

For three events,

\[
P(E_1\cup E_2\cup E_3)\le S_1,
\]

and

\[
P(E_1\cup E_2\cup E_3)\ge S_1-S_2.
\]

The full expression $S_1-S_2+S_3$ gives equality.

In [ ]:
lower_bonferroni = S1 - S2
upper_bonferroni = S1

pd.DataFrame({
    "Bound / probability": ["Lower Bonferroni S1-S2", "True union probability", "Upper Bonferroni S1"],
    "value": [lower_bonferroni, p_union, upper_bonferroni]
})

### Solution

The simulated values should satisfy

\[
S_1-S_2
\le
P(E_1\cup E_2\cup E_3)
\le
S_1.
\]

## 4. Empty Boxes Example

Place $n$ distinguishable balls independently and uniformly into $m$ distinguishable boxes.  
Let $E$ be the event that at least one box is empty.

For box $\ell$,

\[
P(E_\ell)=\left(1-\frac1m\right)^n.
\]

By the union bound,

\[
P(E)\le m\left(1-\frac1m\right)^n.
\]

The exact probability is

\[
P(E)
=
\sum_{j=1}^{m}(-1)^{j-1}\binom{m}{j}
\left(\frac{m-j}{m}\right)^n.
\]

In [ ]:
def empty_box_exact(n, m):
    return sum(((-1)**(j-1)) * comb(m, j) * ((m-j)/m)**n for j in range(1, m+1))

def empty_box_union_bound(n, m):
    return m * (1 - 1/m)**n

m = 10
ns = np.arange(10, 101, 5)
exact = np.array([empty_box_exact(n, m) for n in ns])
bound = np.array([empty_box_union_bound(n, m) for n in ns])

plt.figure(figsize=(7, 4))
plt.plot(ns, exact, marker="o", label="Exact P(at least one empty box)")
plt.plot(ns, bound, marker="s", label="Union bound")
plt.xlabel("number of balls n")
plt.ylabel("Probability / bound")
plt.title("Empty Boxes: Exact Probability vs Union Bound")
plt.legend()
plt.show()

pd.DataFrame({"n": ns, "exact": exact, "union_bound": bound}).head(10)

### Solution

The event that at least one box is empty is

\[
E=E_1\cup\cdots\cup E_m.
\]

The union bound gives

\[
P(E)
\le
\sum_{\ell=1}^m P(E_\ell)
=
m\left(1-\frac1m\right)^n.
\]

The exact expression follows from inclusion-exclusion.

## 5. Markov's Inequality

If $X\ge 0$, then for $a>0$,

\[
P(X\ge a)\le \frac{E[X]}{a}.
\]

We compare the true tail probability and the Markov bound for an exponential random variable.

In [ ]:
lam = 1.0
a_grid = np.linspace(0.2, 8, 200)

true_tail = np.exp(-lam * a_grid)
markov_bound = (1/lam) / a_grid
markov_bound_clipped = np.minimum(markov_bound, 1)

plt.figure(figsize=(7, 4))
plt.plot(a_grid, true_tail, label="True P(X ≥ a), X~Exp(1)")
plt.plot(a_grid, markov_bound_clipped, label="Markov bound min(1, E[X]/a)")
plt.xlabel("a")
plt.ylabel("Probability")
plt.title("Markov Bound for Exponential Tail")
plt.legend()
plt.show()

### Solution

For $X\sim\mathrm{Exponential}(1)$,

\[
E[X]=1,
\qquad
P(X\ge a)=e^{-a}.
\]

Markov gives

\[
P(X\ge a)\le \frac1a.
\]

This is valid but often loose.

## 6. Generalized Markov Inequality

If $g(X)\ge 0$, then

\[
P(g(X)\ge a)\le \frac{E[g(X)]}{a}.
\]

A common choice is $g(X)=|X|^r$, giving

\[
P(|X|\ge a)\le \frac{E[|X|^r]}{a^r}.
\]

In [ ]:
# Compare first-moment and fourth-moment Markov bounds for a standard normal.
a_grid = np.linspace(0.5, 5, 200)

# True two-sided normal tail
true_tail = 2 * (1 - stats.norm.cdf(a_grid))

# First absolute moment E|Z| = sqrt(2/pi)
first_moment_bound = np.sqrt(2/np.pi) / a_grid

# Fourth moment E[Z^4] = 3
fourth_moment_bound = 3 / (a_grid**4)

plt.figure(figsize=(7, 4))
plt.plot(a_grid, true_tail, label="True P(|Z| ≥ a)")
plt.plot(a_grid, np.minimum(first_moment_bound, 1), label="Markov with E|Z|")
plt.plot(a_grid, np.minimum(fourth_moment_bound, 1), label="Markov with E[Z^4]")
plt.yscale("log")
plt.xlabel("a")
plt.ylabel("Tail probability / bound")
plt.title("Generalized Markov Bounds for Standard Normal")
plt.legend()
plt.show()

### Solution

For $Z\sim N(0,1)$,

\[
P(|Z|\ge a)=P(|Z|^r\ge a^r)
\le \frac{E|Z|^r}{a^r}.
\]

For $r=4$,

\[
E[Z^4]=3,
\]

so

\[
P(|Z|\ge a)\le \frac{3}{a^4}.
\]

## 7. Chebyshev's Inequality

If $X$ has mean $\mu$ and variance $\sigma^2$, then

\[
P(|X-\mu|\ge a)\le \frac{\sigma^2}{a^2}.
\]

Equivalently,

\[
P(|X-\mu|\ge k\sigma)\le \frac1{k^2}.
\]

In [ ]:
# Heavy-tailed example: Student t with df=3 has finite variance = 3.
df = 3
mu = 0
var = df / (df - 2)

a_grid = np.linspace(0.5, 10, 200)
true_tail = 2 * (1 - stats.t.cdf(a_grid, df=df))
cheb_bound = var / (a_grid**2)

plt.figure(figsize=(7, 4))
plt.plot(a_grid, true_tail, label="True tail: t(df=3)")
plt.plot(a_grid, np.minimum(cheb_bound, 1), label="Chebyshev bound")
plt.yscale("log")
plt.xlabel("a")
plt.ylabel("Probability / bound")
plt.title("Chebyshev Bound for a Heavy-Tailed Distribution")
plt.legend()
plt.show()

### Solution

For $T\sim t_3$,

\[
E[T]=0,\qquad \operatorname{Var}(T)=\frac{3}{3-2}=3.
\]

Thus

\[
P(|T|\ge a)\le \frac{3}{a^2}.
\]

Chebyshev requires only the variance, so it works broadly but may be conservative.

## 8. Chebyshev for Sample Means

Let $X_1,\ldots,X_n$ be iid with variance $\sigma^2$. Then

\[
\operatorname{Var}(\bar X)=\frac{\sigma^2}{n}.
\]

Therefore,

\[
P(|\bar X-\mu|\ge \epsilon)
\le
\frac{\sigma^2}{n\epsilon^2}.
\]

This shows convergence in probability of $\bar X$ to $\mu$.

In [ ]:
# Example: Bernoulli(p) sample mean
p = 0.4
sigma2 = p * (1 - p)
eps = 0.10

n_values = np.arange(10, 1001, 10)
cheb = sigma2 / (n_values * eps**2)

# True probability for Binomial sample mean
true_probs = []
for n in n_values:
    # P(|Xbar-p| >= eps)
    low = int(np.floor(n * (p - eps)))
    high = int(np.ceil(n * (p + eps)))
    prob_low = stats.binom.cdf(low, n, p)
    prob_high = 1 - stats.binom.cdf(high - 1, n, p)
    true_probs.append(prob_low + prob_high)
true_probs = np.array(true_probs)

plt.figure(figsize=(7, 4))
plt.plot(n_values, true_probs, label="Exact probability")
plt.plot(n_values, np.minimum(cheb, 1), label="Chebyshev bound")
plt.xlabel("n")
plt.ylabel(r"P(|Xbar-p| ≥ epsilon)")
plt.title("Chebyshev Bound for Bernoulli Sample Mean")
plt.legend()
plt.show()

### Solution

Since

\[
E[\bar X]=p,\qquad \operatorname{Var}(\bar X)=\frac{p(1-p)}{n},
\]

Chebyshev gives

\[
P(|\bar X-p|\ge \epsilon)
\le
\frac{p(1-p)}{n\epsilon^2}.
\]

The bound decreases like $1/n$.

## 9. Chernoff Bound for an Exponential Random Variable

For any $t>0$,

\[
P(X\ge a)
=
P(e^{tX}\ge e^{ta})
\le
\frac{E[e^{tX}]}{e^{ta}}.
\]

If $X\sim \mathrm{Exponential}(\lambda)$, then

\[
E[e^{tX}]=\frac{\lambda}{\lambda-t},
\qquad 0<t<\lambda.
\]

Thus

\[
P(X\ge a)\le
\frac{\lambda}{\lambda-t}e^{-ta}.
\]

The best $t$ is $t^*=\lambda-\frac1a$ when $a>1/\lambda$.

In [ ]:
lam = 1.0
a_grid = np.linspace(1.05, 8, 200)

true_tail = np.exp(-lam * a_grid)
markov = 1 / a_grid
cheb = 1 / ((a_grid - 1)**2)  # Chebyshev around mean 1, variance 1; valid for a>1
chernoff = (lam * a_grid) * np.exp(1 - lam * a_grid)  # optimized

plt.figure(figsize=(7, 4))
plt.plot(a_grid, true_tail, label="True exp(-a)")
plt.plot(a_grid, np.minimum(markov, 1), label="Markov")
plt.plot(a_grid, np.minimum(cheb, 1), label="Chebyshev")
plt.plot(a_grid, np.minimum(chernoff, 1), label="Chernoff")
plt.yscale("log")
plt.xlabel("a")
plt.ylabel("Tail probability / bound")
plt.title("Tail Bounds for X ~ Exp(1)")
plt.legend()
plt.show()

### Solution

For $X\sim\mathrm{Exp}(\lambda)$,

\[
P(X\ge a)
\le
\frac{\lambda}{\lambda-t}e^{-ta}.
\]

Minimize the log-bound:

\[
\log\left(\frac{\lambda}{\lambda-t}e^{-ta}\right)
=
\log \lambda-\log(\lambda-t)-ta.
\]

Differentiate:

\[
\frac{1}{\lambda-t}-a=0.
\]

So

\[
t^*=\lambda-\frac1a.
\]

Substitute this into the bound:

\[
P(X\ge a)
\le
\lambda a e^{1-\lambda a}.
\]

The true tail is $e^{-\lambda a}$, so Chernoff has the correct exponential decay rate.

## 10. Chernoff Bound for a Bernoulli Sample Mean

Let $X_i\sim \mathrm{Bernoulli}(p)$ independently and

\[
\bar X=\frac1n\sum_{i=1}^n X_i.
\]

For $q>p$, a Chernoff bound is

\[
P(\bar X\ge q)
\le
\exp\{-nD(q\Vert p)\},
\]

where

\[
D(q\Vert p)
=
q\log\frac{q}{p}
+
(1-q)\log\frac{1-q}{1-p}.
\]

In [ ]:
def kl_bernoulli(q, p):
    return q*np.log(q/p) + (1-q)*np.log((1-q)/(1-p))

p = 0.5
q = 0.75
n_values = np.arange(10, 301, 10)

chernoff = np.exp(-n_values * kl_bernoulli(q, p))
cheb = p*(1-p) / (n_values * (q-p)**2)

exact = []
for n in n_values:
    cutoff = int(np.ceil(n*q))
    exact.append(1 - stats.binom.cdf(cutoff - 1, n, p))
exact = np.array(exact)

plt.figure(figsize=(7, 4))
plt.plot(n_values, exact, label="Exact P(Xbar ≥ q)")
plt.plot(n_values, chernoff, label="Chernoff bound")
plt.plot(n_values, np.minimum(cheb, 1), label="Chebyshev bound")
plt.yscale("log")
plt.xlabel("n")
plt.ylabel("Probability / bound")
plt.title("Bernoulli Sample Mean: Chernoff vs Chebyshev")
plt.legend()
plt.show()

### Solution

For Bernoulli sample means, Chernoff gives an exponential bound:

\[
P(\bar X\ge q)
\le
e^{-nD(q\Vert p)}.
\]

Chebyshev gives only a polynomial bound:

\[
P(\bar X-p\ge q-p)
\le
\frac{p(1-p)}{n(q-p)^2}.
\]

This is why Chernoff bounds are important for large-sample and high-dimensional probability.

## 11. Hoeffding's Inequality

If $0\le X_i\le 1$ are independent, then

\[
P(|\bar X-E\bar X|\ge \epsilon)
\le
2e^{-2n\epsilon^2}.
\]

This is an exponential concentration bound.

In [ ]:
p = 0.4
eps = 0.10
n_values = np.arange(10, 1001, 10)

hoeffding = 2*np.exp(-2*n_values*eps**2)
chebyshev = p*(1-p)/(n_values*eps**2)

exact = []
for n in n_values:
    low = int(np.floor(n*(p-eps)))
    high = int(np.ceil(n*(p+eps)))
    prob_low = stats.binom.cdf(low, n, p)
    prob_high = 1 - stats.binom.cdf(high - 1, n, p)
    exact.append(prob_low + prob_high)
exact = np.array(exact)

plt.figure(figsize=(7, 4))
plt.plot(n_values, exact, label="Exact")
plt.plot(n_values, np.minimum(hoeffding, 1), label="Hoeffding")
plt.plot(n_values, np.minimum(chebyshev, 1), label="Chebyshev")
plt.yscale("log")
plt.xlabel("n")
plt.ylabel(r"P(|Xbar-p| ≥ epsilon)")
plt.title("Hoeffding vs Chebyshev for Bernoulli Means")
plt.legend()
plt.show()

### Solution

For Bernoulli variables, $0\le X_i\le 1$, so Hoeffding applies:

\[
P(|\bar X-p|\ge \epsilon)
\le 2e^{-2n\epsilon^2}.
\]

This often improves dramatically over Chebyshev for large $n$ because it decays exponentially.

## 12. Cauchy--Schwarz Inequality

For random variables $X,Y$,

\[
(E[XY])^2\le E[X^2]E[Y^2].
\]

As a consequence,

\[
|\operatorname{Corr}(X,Y)|\le 1.
\]

In [ ]:
N = 300_000
X = rng.normal(0, 1, size=N)
Y = 2*X + rng.normal(0, 1, size=N)

EXY = np.mean(X*Y)
EX2 = np.mean(X**2)
EY2 = np.mean(Y**2)

lhs = EXY**2
rhs = EX2 * EY2
corr = np.corrcoef(X, Y)[0, 1]

print("(E[XY])^2:", lhs)
print("E[X^2]E[Y^2]:", rhs)
print("Cauchy-Schwarz holds:", lhs <= rhs)
print("Correlation:", corr)
print("|Correlation| <= 1:", abs(corr) <= 1)

### Solution

Let

\[
U=\frac{X-E[X]}{\sqrt{\operatorname{Var}(X)}},
\qquad
V=\frac{Y-E[Y]}{\sqrt{\operatorname{Var}(Y)}}.
\]

Then

\[
\operatorname{Corr}(X,Y)=E[UV].
\]

By Cauchy--Schwarz,

\[
|E[UV]|
\le
\sqrt{E[U^2]E[V^2]}
=
1.
\]

Therefore,

\[
|\operatorname{Corr}(X,Y)|\le 1.
\]

## 13. Hölder's Inequality

If $p,q>1$ and

\[
\frac1p+\frac1q=1,
\]

then

\[
E[|XY|]\le
\left(E[|X|^p]\right)^{1/p}
\left(E[|Y|^q]\right)^{1/q}.
\]

Cauchy--Schwarz is the special case $p=q=2$.

In [ ]:
N = 300_000
X = rng.normal(0, 1, size=N)
Y = rng.exponential(scale=1, size=N) - 1

p_val = 3
q_val = 3/2

left = np.mean(np.abs(X*Y))
right = (np.mean(np.abs(X)**p_val))**(1/p_val) * (np.mean(np.abs(Y)**q_val))**(1/q_val)

print("E|XY|:", left)
print("Holder bound:", right)
print("Inequality holds:", left <= right)

### Solution

For $p=3$ and $q=3/2$,

\[
\frac13+\frac{2}{3}=1.
\]

Therefore, Hölder's inequality gives

\[
E[|XY|]
\le
(E[|X|^3])^{1/3}(E[|Y|^{3/2}])^{2/3}.
\]

The simulation verifies the inequality numerically.

## 14. Minkowski's Inequality

For $p\ge 1$,

\[
\left(E[|X+Y|^p]\right)^{1/p}
\le
\left(E[|X|^p]\right)^{1/p}
+
\left(E[|Y|^p]\right)^{1/p}.
\]

This is the triangle inequality for $L^p$ norms.

In [ ]:
N = 300_000
X = rng.normal(0, 1, size=N)
Y = rng.normal(1, 2, size=N)

p_val = 4

norm_X_plus_Y = (np.mean(np.abs(X+Y)**p_val))**(1/p_val)
norm_X = (np.mean(np.abs(X)**p_val))**(1/p_val)
norm_Y = (np.mean(np.abs(Y)**p_val))**(1/p_val)

print("||X+Y||_p:", norm_X_plus_Y)
print("||X||_p + ||Y||_p:", norm_X + norm_Y)
print("Minkowski holds:", norm_X_plus_Y <= norm_X + norm_Y)

### Solution

The $L^p$ norm of a random variable is

\[
\|X\|_p=(E|X|^p)^{1/p}.
\]

Minkowski says

\[
\|X+Y\|_p\le \|X\|_p+\|Y\|_p.
\]

The simulation verifies the inequality for $p=4$.

## 15. Jensen's Inequality

If $g$ is convex, then

\[
g(E[X])\le E[g(X)].
\]

If $g$ is concave, the inequality reverses.

Examples:

\[
(E[X])^2\le E[X^2],
\]

because $g(x)=x^2$ is convex.

Also,

\[
E[\log X]\le \log E[X],
\]

because $\log x$ is concave.

In [ ]:
N = 300_000
X = rng.exponential(scale=2, size=N)

# Convex g(x)=x^2
left_convex = X.mean()**2
right_convex = np.mean(X**2)

# Concave g(x)=log(x)
left_concave = np.mean(np.log(X))
right_concave = np.log(np.mean(X))

print("Convex example: (E[X])^2 =", left_convex)
print("Convex example: E[X^2] =", right_convex)
print("(E[X])^2 <= E[X^2]:", left_convex <= right_convex)

print()
print("Concave example: E[log X] =", left_concave)
print("Concave example: log E[X] =", right_concave)
print("E[log X] <= log E[X]:", left_concave <= right_concave)

In [ ]:
# Visualization: convex function and tangent line
x_grid = np.linspace(-3, 3, 400)
g = x_grid**2

x0 = 1.0
tangent = x0**2 + 2*x0*(x_grid - x0)

plt.figure(figsize=(7, 4))
plt.plot(x_grid, g, label=r"$g(x)=x^2$")
plt.plot(x_grid, tangent, linestyle="--", label="tangent at x=1")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Convex Function Lies Above Its Tangent")
plt.legend()
plt.show()

### Solution

For convex $g$,

\[
g(E[X])\le E[g(X)].
\]

Taking $g(x)=x^2$ gives

\[
(E[X])^2\le E[X^2],
\]

which is another way to see that

\[
\operatorname{Var}(X)=E[X^2]-(E[X])^2\ge 0.
\]

For concave $g(x)=\log x$,

\[
E[\log X]\le \log E[X].
\]

# Practice Problems with Full Solutions

## Practice Problem 1 — Markov Bound

Let $X\sim\mathrm{Poisson}(4)$. Use Markov's inequality to bound

\[
P(X\ge 10).
\]

Then compute the exact probability.

In [ ]:
lam = 4
a = 10

markov = lam / a
exact = 1 - stats.poisson.cdf(a-1, lam)

print("Markov bound:", markov)
print("Exact probability:", exact)

### Solution

Since $X\ge 0$ and $E[X]=4$,

\[
P(X\ge 10)\le \frac{E[X]}{10}=\frac{4}{10}=0.4.
\]

The exact probability is

\[
P(X\ge 10)=1-P(X\le 9),
\]

which Python computes using the Poisson CDF.

## Practice Problem 2 — Chebyshev Bound

Let $X$ have mean $50$ and standard deviation $5$. Bound

\[
P(|X-50|\ge 15).
\]

In [ ]:
mu = 50
sigma = 5
a = 15

cheb_bound = sigma**2 / a**2
print("Chebyshev bound:", cheb_bound)

### Solution

Chebyshev says

\[
P(|X-\mu|\ge a)\le \frac{\sigma^2}{a^2}.
\]

Here $\mu=50$, $\sigma=5$, and $a=15$. Therefore,

\[
P(|X-50|\ge 15)
\le
\frac{5^2}{15^2}
=
\frac{25}{225}
=
\frac19
\approx 0.1111.
\]

## Practice Problem 3 — Hoeffding Sample Size

Suppose $X_1,\ldots,X_n$ are iid Bernoulli random variables with mean $p$.  
Find $n$ such that

\[
P(|\bar X-p|\ge 0.05)\le 0.01
\]

using Hoeffding's inequality.

In [ ]:
eps = 0.05
delta = 0.01

n_required = np.ceil(np.log(2/delta)/(2*eps**2))
print("Required n:", int(n_required))

bound = 2*np.exp(-2*n_required*eps**2)
print("Hoeffding bound at this n:", bound)

### Solution

Hoeffding gives

\[
P(|\bar X-p|\ge \epsilon)\le 2e^{-2n\epsilon^2}.
\]

We need

\[
2e^{-2n(0.05)^2}\le 0.01.
\]

Divide by $2$ and take logs:

\[
-2n(0.05)^2\le \log(0.005).
\]

Thus

\[
n\ge
\frac{\log(2/0.01)}{2(0.05)^2}.
\]

Python computes the ceiling of this value.

## Practice Problem 4 — Chernoff Bound for Bernoulli

Let $X_1,\ldots,X_n\sim\mathrm{Bernoulli}(0.5)$ with $n=100$.  
Use the Chernoff bound to estimate

\[
P(\bar X\ge 0.65).
\]

Compare with the exact probability.

In [ ]:
p = 0.5
q = 0.65
n = 100

D = kl_bernoulli(q, p)
chernoff_bound = np.exp(-n*D)

cutoff = int(np.ceil(n*q))
exact_prob = 1 - stats.binom.cdf(cutoff - 1, n, p)

print("KL D(q||p):", D)
print("Chernoff bound:", chernoff_bound)
print("Exact probability:", exact_prob)

### Solution

For $q>p$,

\[
P(\bar X\ge q)\le e^{-nD(q\Vert p)}.
\]

Here $p=0.5$, $q=0.65$, and $n=100$, so

\[
D(0.65\Vert0.5)
=
0.65\log\frac{0.65}{0.5}
+
0.35\log\frac{0.35}{0.5}.
\]

Then

\[
P(\bar X\ge 0.65)\le e^{-100D(0.65\Vert0.5)}.
\]

The exact probability is computed from the binomial distribution:

\[
P(\bar X\ge 0.65)=P(S\ge 65),
\qquad S\sim\mathrm{Binomial}(100,0.5).
\]

## Practice Problem 5 — Cauchy--Schwarz and Correlation

Let $X\sim N(0,1)$ and $Y=3X+2$. Show that $|\operatorname{Corr}(X,Y)|=1$ by simulation.

In [ ]:
N = 100_000
X = rng.normal(0, 1, size=N)
Y = 3*X + 2

corr = np.corrcoef(X, Y)[0, 1]
print("Correlation:", corr)
print("Absolute correlation:", abs(corr))

### Solution

Since

\[
Y=3X+2,
\]

$Y$ is an exact linear function of $X$. Equality in the correlation bound occurs when

\[
Y=a+bX
\]

for constants $a,b$ with $b\ne 0$. Hence

\[
|\operatorname{Corr}(X,Y)|=1.
\]

Since $b=3>0$, the correlation is $+1$.

## Practice Problem 6 — Jensen's Inequality

Let $X\sim\mathrm{Uniform}(0,1)$. Verify that

\[
E[e^X]\ge e^{E[X]}.
\]

In [ ]:
N = 300_000
X = rng.uniform(0, 1, size=N)

lhs = np.mean(np.exp(X))
rhs = np.exp(np.mean(X))

theory_lhs = np.e - 1
theory_rhs = np.exp(0.5)

print("Simulation E[e^X]:", lhs)
print("Simulation e^{E[X]}:", rhs)
print("Theory E[e^X] = e-1:", theory_lhs)
print("Theory e^{1/2}:", theory_rhs)

### Solution

The function $g(x)=e^x$ is convex. Jensen's inequality gives

\[
e^{E[X]}\le E[e^X].
\]

For $X\sim \mathrm{Uniform}(0,1)$,

\[
E[X]=\frac12,
\]

and

\[
E[e^X]=\int_0^1 e^x\,dx=e-1.
\]

Thus

\[
e^{1/2}\le e-1.
\]

# Summary

This lab used simulation and computation to study the main inequalities in Section 7.

Key takeaways:

- **Boole's inequality** bounds union probabilities.
- **Inclusion-exclusion** gives exact union probabilities when all intersections are known.
- **Bonferroni inequalities** provide partial inclusion-exclusion bounds.
- **Markov's inequality** uses only a first moment.
- **Chebyshev's inequality** uses mean and variance.
- **Chernoff bounds** use moment generating functions and often give exponential bounds.
- **Hoeffding's inequality** gives exponential concentration for bounded independent variables.
- **Cauchy--Schwarz, Hölder, and Minkowski** are norm and inner-product inequalities.
- **Jensen's inequality** connects convexity and expectation.

\[
\boxed{\text{Inequalities let us control probabilities even when exact distributions are hard.}}
\]